# Train the “Hey DJ” wake word for GriffTunes

This makes **hey_dj.onnx**, a small (about 200 KB) wake word model that GriffTunes runs right in Chrome on the jukebox.
It's openWakeWord's automatic training (github.com/dscripka/openWakeWord): thousands of made-up voices say “hey DJ”
(and things that sound close to it), get mixed with room echo and background music, and a tiny model learns to tell them apart.
No recordings of anyone in the house are needed.

**How to run it (about an hour, nothing to type):**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** If Colab warns that the notebook isn't from Google, choose **Run anyway**.
3. Leave the tab open. When it's done, the last cell downloads **hey_dj.onnx** (Chrome may ask to allow the download).
4. Send hey_dj.onnx to Claude in the GriffTunes project, and it goes into the jukebox.

If a cell stops with a red error, send Claude a screenshot of the last lines. Running **Run all** again picks up where it left off.

Everything heavy runs in its own Python (made with `uv`, with versions that were tested together on Nov 14, 2026), so Colab's own updates don't break it.

In [ ]:
# 1. Check for the GPU (it works without one, but generating the voices takes many hours on CPU)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: use Runtime > Change runtime type > T4 GPU, then Run all again"

In [ ]:
# 2. Install everything into its own Python 3.12 (about 3 minutes)
%cd /content
!pip install -q uv
!test -x /content/venv/bin/python || uv venv -q -p 3.12 /content/venv
!uv pip install -q --python /content/venv/bin/python \
  torch==2.5.1 torchaudio==2.5.1 numpy==1.26.4 scipy==1.13.1 setuptools==69.5.1 \
  piper-phonemize-cross==1.2.1 webrtcvad==2.0.10 mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
  speechbrain==0.5.16 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 \
  pronouncing==0.2.0 deep-phonemizer==0.0.19 onnxruntime==1.32.0 onnx==1.23.2 scikit-learn==1.9.1 \
  pyyaml==6.0.3 soundfile==0.14.0 tqdm==4.70.1 requests==2.34.2 \
  datasets==2.14.6 pyarrow==17.0.0 pandas==2.2.3 librosa==0.10.2.post1 huggingface_hub==0.24.7
# openWakeWord (the exact version this notebook was tested with) and the Piper voice generator it expects
!test -d openWakeWord || (git clone -q https://github.com/dscripka/openWakeWord && cd openWakeWord && git checkout -q 368c03716d1e92591906a84949bc477f3a834455)
!uv pip install -q --python /content/venv/bin/python --no-deps -e ./openWakeWord
!test -d piper-sample-generator || git clone -q --branch v2.0.0 --depth 1 https://github.com/rhasspy/piper-sample-generator
!test -s piper-sample-generator/models/en_US-libritts_r-medium.pt || wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
!mkdir -p openWakeWord/openwakeword/resources/models
!for f in embedding_model melspectrogram; do test -s openWakeWord/openwakeword/resources/models/$f.onnx || wget -q -O openWakeWord/openwakeword/resources/models/$f.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/$f.onnx; done
# the training script always tries a TFLite export at the end (a flag bug); GriffTunes only needs the ONNX file
!sed -i 's/if args.convert_to_tflite:/if args.convert_to_tflite is True:/' openWakeWord/openwakeword/train.py
!/content/venv/bin/python -c "import torch, openwakeword.data, piper_phonemize; print('Ready. GPU:', torch.cuda.is_available())"

## Background sounds and practice data
Room echo (MIT impulse responses), everyday noise (AudioSet), music (Free Music Archive), and openWakeWord's
precomputed “not the wake word” data (about 2,000 hours of speech and sound, a large download of roughly 17 GB).

In [ ]:
%%writefile /content/get_data.py
import os, sys, subprocess, numpy as np, scipy.io.wavfile as wavfile
from pathlib import Path
from tqdm import tqdm
import datasets
os.chdir('/content')
def save(path, arr):
    wavfile.write(path, 16000, (np.clip(arr, -1, 1) * 32767).astype(np.int16))

# Room echo
os.makedirs('mit_rirs', exist_ok=True)
if len(os.listdir('mit_rirs')) < 200:
    ds = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True)
    for row in tqdm(ds, desc='room echo'):
        save(os.path.join('mit_rirs', row['audio']['path'].split('/')[-1]), row['audio']['array'])
print('room echo clips:', len(os.listdir('mit_rirs')))

# Everyday noise: one part of AudioSet (optional; music below is enough on its own)
os.makedirs('audioset_16k', exist_ok=True)
if len(os.listdir('audioset_16k')) < 100:
    try:
        os.makedirs('audioset', exist_ok=True)
        subprocess.run(['wget', '-q', '-O', 'audioset/bal_train09.tar', 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar'], check=True)
        subprocess.run(['tar', '-xf', 'bal_train09.tar'], cwd='audioset', check=True)
        files = [str(i) for i in Path('audioset').glob('**/*.flac')]
        ad = datasets.Dataset.from_dict({'audio': files}).cast_column('audio', datasets.Audio(sampling_rate=16000))
        for row in tqdm(ad, desc='everyday noise'):
            save(os.path.join('audioset_16k', row['audio']['path'].split('/')[-1].replace('.flac', '.wav')), row['audio']['array'])
    except Exception as e:
        print('Skipping AudioSet (not required):', e)
print('noise clips:', len(os.listdir('audioset_16k')))

# Music: 3 hours of 30-second clips from the Free Music Archive (the jukebox will be playing music)
os.makedirs('fma', exist_ok=True)
want = 3 * 3600 // 30
if len(os.listdir('fma')) < want:
    fm = datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
    fm = iter(fm.cast_column('audio', datasets.Audio(sampling_rate=16000)))
    for _ in tqdm(range(want - len(os.listdir('fma'))), desc='music'):
        try:
            row = next(fm)
        except StopIteration:
            break
        except Exception as e:
            print('skipped a music clip:', e); continue
        save(os.path.join('fma', row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')), row['audio']['array'])
print('music clips:', len(os.listdir('fma')))
if len(os.listdir('fma')) + len(os.listdir('audioset_16k')) < 50:
    sys.exit('Not enough background audio was downloaded. Run this cell again.')

In [ ]:
!/content/venv/bin/python /content/get_data.py
%cd /content
!test -s openwakeword_features_ACAV100M_2000_hrs_16bit.npy || wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!test -s validation_set_features.npy || wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
!ls -la *.npy

## The settings for “Hey DJ”
Piper says the phrase best spelled out as “hey dee jay”. Close-sounding phrases (“hey jay”, “dee jay”, “hey day”…) are made on purpose,
so the model learns *not* to wake up for them.

In [ ]:
%%writefile /content/make_config.py
import yaml
c = yaml.load(open('/content/openWakeWord/examples/custom_model.yml').read(), yaml.Loader)
c.update(
    target_phrase=['hey dee jay'],
    custom_negative_phrases=['hey jay', 'dee jay', 'hey day', 'hey dave', 'hey kay', 'hey there', 'the DJ', 'hey G', 'okay', 'hey hey'],
    model_name='hey_dj',
    n_samples=10000, n_samples_val=2000, tts_batch_size=50, augmentation_batch_size=16,
    piper_sample_generator_path='/content/piper-sample-generator',
    output_dir='/content/my_custom_model',
    rir_paths=['/content/mit_rirs'],
    background_paths=['/content/audioset_16k', '/content/fma'],
    background_paths_duplication_rate=[1, 2],
    false_positive_validation_data_path='/content/validation_set_features.npy',
    feature_data_files={'ACAV100M_sample': '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'},
    steps=40000,
    target_false_positives_per_hour=0.2,
)
yaml.dump(c, open('/content/hey_dj.yaml', 'w'))
print({k: c[k] for k in ['target_phrase', 'n_samples', 'steps']})

In [ ]:
!/content/venv/bin/python /content/make_config.py

## Train (the long part: roughly 20 + 10 + 20 minutes on a T4)

In [ ]:
# Step 1: thousands of made-up voices say "hey dee jay" and close-sounding phrases (re-running continues where it stopped)
%cd /content
!/content/venv/bin/python openWakeWord/openwakeword/train.py --training_config hey_dj.yaml --generate_clips 2>&1 | grep -v -i warning | tail -n 25

In [ ]:
# Step 2: add room echo and background noise/music, then turn the clips into openWakeWord features
%cd /content
!/content/venv/bin/python openWakeWord/openwakeword/train.py --training_config hey_dj.yaml --augment_clips 2>&1 | grep -v -i warning | tail -n 15

In [ ]:
# Step 3: train the model
%cd /content
!/content/venv/bin/python openWakeWord/openwakeword/train.py --training_config hey_dj.yaml --train_model 2>&1 | grep -v -i warning | tail -n 25
!ls -la /content/my_custom_model/hey_dj.onnx

In [ ]:
%%writefile /content/check.py
# A quick check on clips the model never trained on: "hey dee jay" should score high, the look-alikes low
import glob, numpy as np, scipy.io.wavfile as wavfile
from openwakeword.model import Model
def best(path):
    m = Model(wakeword_models=['/content/my_custom_model/hey_dj.onnx'], inference_framework='onnx')
    sr, a = wavfile.read(path)
    a = np.concatenate([np.zeros(16000, np.int16), a.astype(np.int16), np.zeros(16000, np.int16)])
    return max(float(list(m.predict(a[i:i + 1280]).values())[0]) for i in range(0, len(a) - 1279, 1280))
pos = sorted(glob.glob('/content/my_custom_model/hey_dj/positive_test/*.wav'))[:25]
neg = sorted(glob.glob('/content/my_custom_model/hey_dj/negative_test/*.wav'))[:25]
p = [best(f) for f in pos]; n = [best(f) for f in neg]
print(f'"hey dee jay" clips heard (score >= 0.5): {sum(x >= .5 for x in p)} of {len(p)}')
print(f'look-alike clips that would wake it:     {sum(x >= .5 for x in n)} of {len(n)}')

In [ ]:
!cd /content && /content/venv/bin/python /content/check.py 2>&1 | grep -v -i warning | tail -n 3

In [ ]:
# Download the model: send this file to Claude in the GriffTunes project
from google.colab import files
files.download('/content/my_custom_model/hey_dj.onnx')